## Big Project for PDFA module
Author Cathal Redmond
Date 22 Dec 2025. (Ho Ho Ho.)


In [3]:
## Big Project 
## I am opting to do the suggested wind farm data analysis project.

In [4]:
##imports
import numpy as np
import pandas as pd

import zipfile


In [5]:
#define the URLs of the zip files
johnstown = ("https://cli.fusio.net/cli/climate_data/webdata/hly1775.zip") 

In [6]:
# pandas support zip file reads   # reference https://stackoverflow.com/questions/26942476/reading-csv-zipped-files-in-python
#dfjohnstown = pd.read_csv("zip://" + johnstown)

In [1]:
from csv import DictReader
from io import TextIOWrapper, BytesIO
from zipfile import ZipFile

import requests
import sqlite3

def all_tickers():
    url = "https://cli.fusio.net/cli/climate_data/webdata/hly1775.zip"
    r = requests.get(url)
    zip_ref = ZipFile(BytesIO(r.content))
    
    # Connect to SQLite database
    conn = sqlite3.connect('climate_data.db')
    cursor = conn.cursor()
    
    for name in zip_ref.namelist():
        if name.endswith('.csv'):  # Only process CSV files
            print(f"Processing {name}")
            with zip_ref.open(name) as file_contents:
                reader = DictReader(TextIOWrapper(file_contents, 'latin-1'), delimiter=';')
                
                # Get column names from the first row
                headers = reader.fieldnames
                if headers:
                    # Create table dynamically based on headers
                    table_name = name.replace('.csv', '').replace('-', '_').replace(' ', '_')
                    columns = ', '.join([f'"{h}" TEXT' for h in headers])
                    create_table_sql = f'CREATE TABLE IF NOT EXISTS {table_name} ({columns})'
                    cursor.execute(create_table_sql)
                    
                    # Prepare insert statement
                    placeholders = ', '.join(['?' for _ in headers])
                    insert_sql = f'INSERT INTO {table_name} VALUES ({placeholders})'
                    
                    # Insert data
                    for row in reader:
                        values = [row.get(h, '') for h in headers]
                        cursor.execute(insert_sql, values)
                    
                    print(f"Inserted data into table {table_name}")
    
    # Commit and close
    conn.commit()
    conn.close()
    print("Data written to database successfully")

# place the data into a dataframe
# this will be useful as we will need to work with the data. 

all_tickers()
#print(dfjohnstown.head())

Processing hly1775.csv
Inserted data into table hly1775
Data written to database successfully
